# 小恐龍自動遊玩機器人（Chrome Dino Bot）

本程式使用 `pyautogui` 自動偵測 Chrome 小恐龍遊戲中的障礙物，並根據障礙物類型與遊戲速度自動做出跳躍或蹲下的動作。

## 遊戲機制說明
- **仙人掌（Cactus）**：地面障礙物，必須跳躍躲避
- **翼手龍（Pterodactyl）**：空中障礙物
  - 低空翼手龍：需跳躍躲避
  - 高空翼手龍：需蹲下躲避
- **遊戲速度**：隨著遊玩時間增加，`Runner.instance_.currentSpeed` 會不斷提升，因此跳躍時機必須動態調整

In [70]:
# -*- coding: utf-8 -*-
# 導入必要的套件
import time
import numpy as np
import pyautogui as pyg
from PIL import ImageGrab
import cv2

# 設定 pyautogui 的安全機制
# 將滑鼠移動到螢幕左上角（(0,0)）可緊急停止程式
pyg.FAILSAFE = True
# 設定動作延遲為零，加快反應速度
pyg.PAUSE = 0.001

print("套件載入完成！")
print("提示：將滑鼠移動至螢幕左上角可緊急中斷程式")

套件載入完成！
提示：將滑鼠移動至螢幕左上角可緊急中斷程式


## 參數設定

在開始遊玩前，需要根據自己的螢幕解析度與遊戲視窗位置調整以下參數。

In [71]:
# ==================== 遊戲區域設定 ====================
# 請根據實際情況調整這些座標
# 建議先用截圖找出正確的座標值

# 恐龍位置（用於參考）
DINO_X = 53
DINO_Y = 382

# 偵測區域設定：在恐龍前方掃描障礙物
# 偵測區域的左上角座標與寬高
DETECT_REGION_LEFT = 200    # 從恐龍右側開始偵測的 X 座標
DETECT_REGION_TOP = 479    # 偵測區域的 Y 座標（地面高度附近）
DETECT_REGION_WIDTH = 660  # 偵測區域寬度（往前方掃描的距離）
DETECT_REGION_HEIGHT = 360 # 偵測區域高度（涵蓋地面到高空）

# ==================== 顏色偵測設定 ====================
# Chrome 小恐龍遊戲在夜間模式或日間模式顏色會不同
# 這裡主要偵測障礙物（通常是深色或白色，視主題而定）
# 一般日間模式：障礙物顏色較深，背景為白色
OBSTACLE_THRESHOLD = 100   # 顏色閾值：低於此值視為障礙物（深色物件）

# ==================== 跳躍與蹲下參數 ====================
# 基礎跳躍距離（像素）- 會根據遊戲速度動態調整
BASE_JUMP_DISTANCE = 120
# 速度調整係數：速度越快，需要越早跳躍
SPEED_FACTOR = 1.5

# 高空翼手龍偵測高度閾值（Y 座標越小代表越高）
# 若障礙物出現在較高的位置，則判定為需蹲下的高空翼手龍
HIGH_PTERODACTYL_Y = 102 

print("參數設定完成！")

參數設定完成！


## 核心偵測與控制函式

以下函式會截取遊戲畫面、分析障礙物、並根據情況執行跳躍或蹲下動作。

In [72]:
# ==================== 核心函式 ====================

def get_game_screenshot():
    """
    截取遊戲畫面
    
    說明：使用 pyautogui 截取整個螢幕，後續會從中擷取偵測區域進行分析
    
    回傳：PIL Image 物件，代表當前遊戲畫面
    """
    screenshot = pyg.screenshot()
    return screenshot


def detect_obstacles(screenshot):
    """
    偵測前方區域中的障礙物（仙人掌與翼手龍）
    
    邏輯說明：
    1. 將截圖轉換為 OpenCV 格式以便進行圖像處理
    2. 擷取恐龍前方的偵測區域
    3. 將圖像轉換為灰階，找出與背景不同的物件（障礙物）
    4. 使用輪廓偵測找出所有障礙物
    5. 分析障礙物的位置判斷類型（地面/高空）
    
    參數：
        screenshot: PIL Image 格式的遊戲截圖
    
    回傳：
        list: 包含障礙物資訊的列表，每個元素為 dict，包含：
              {'x': 水平距離, 'y': 垂直位置, 'width': 寬度, 'height': 高度, 'type': 類型}
    """
    # 將 PIL Image 轉換為 OpenCV 格式（BGR）
    img = np.array(screenshot)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    
    # 擷取偵測區域（恐龍前方）
    detect_area = img[
        DETECT_REGION_TOP:DETECT_REGION_TOP + DETECT_REGION_HEIGHT,
        DETECT_REGION_LEFT:DETECT_REGION_LEFT + DETECT_REGION_WIDTH
    ]
    
    # 轉換為灰階圖像，方便偵測物件
    gray = cv2.cvtColor(detect_area, cv2.COLOR_BGR2GRAY)
    
    # 使用二值化找出障礙物（與背景有明顯差異的區域）
    # 適用於大多數 Chrome Dino 主題
    _, thresh = cv2.threshold(gray, OBSTACLE_THRESHOLD, 255, cv2.THRESH_BINARY_INV)
    
    # 找出輪廓（障礙物的邊界）
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    obstacles = []
    # 分析每個輪廓，判斷是否為障礙物
    for contour in contours:
        # 計算輪廓的邊界框
        x, y, w, h = cv2.boundingRect(contour)
        
        # 過濾太小的雜訊（避免誤判）
        if w < 5 or h < 5:
            continue
        
        # 計算障礙物距離恐龍的水平距離（像素）
        obstacle_distance = x  # 相對於偵測區域左邊緣的距離
        
        # 判斷障礙物類型
        # y 座標越小表示在畫面中越靠上（高空）
        # 翼手龍通常出現在較高的位置
        if y < HIGH_PTERODACTYL_Y - DETECT_REGION_TOP:
            # 高空區域的物件很可能是需要蹲下的翼手龍
            obstacle_type = 'pterodactyl_high'  # 高空翼手龍 -> 需蹲下
        elif y < (DETECT_REGION_HEIGHT // 2):
            # 中高空區域，可能是低空翼手龍
            obstacle_type = 'pterodactyl_low'   # 低空翼手龍 -> 需跳躍
        else:
            # 地面區域，通常是仙人掌
            obstacle_type = 'cactus'            # 仙人掌 -> 需跳躍
        
        obstacles.append({
            'x': obstacle_distance,      # 距離（像素）
            'y': y,                      # 垂直位置
            'width': w,
            'height': h,
            'type': obstacle_type
        })
    
    # 按照距離排序（最近的障礙物優先處理）
    obstacles.sort(key=lambda obs: obs['x'])
    
    return obstacles


def get_game_speed():
    """
    嘗試獲取遊戲當前速度
    
    說明：
    Chrome 小恐龍遊戲的速度儲存在 JavaScript 物件 Runner.instance_.currentSpeed 中。
    但由於我們使用 pyautogui 自動化，無法直接存取瀏覽器的 JS 變數。
    因此改用「時間推估」的方式：遊戲速度會隨著遊玩時間線性提升。
    
    實際使用時，建議觀察遊戲速度並調整參數，或透過瀏覽器擴充功能配合使用。
    
    回傳：
        float: 估算的遊戲速度（像素/更新循環）
    """
    # 計算遊玩時間來估算速度
    # Chrome Dino 的速度大約從 6 開始，每隔一段時間增加
    elapsed_time = time.time() - game_start_time
    
    # 速度估算公式（根據 Chrome Dino 的大致機制）
    # 初始速度約 6，隨時間增加，最高約 13
    estimated_speed = 6 + min(elapsed_time / 500, 7)
    
    return estimated_speed


def calculate_jump_distance(speed):
    """
    根據遊戲速度動態計算應該在何處開始跳躍
    
    說明：
    遊戲速度越快，恐龍的移動速度越快，需要越早（更遠的距離）就開始跳躍
    才能有足夠的時間讓恐龍跳過障礙物。
    
    參數：
        speed: 當前遊戲速度（由 get_game_speed() 估算）
    
    回傳：
        float: 建議的跳躍觸發距離（像素）
    """
    # 動態調整跳躍距離：基礎距離 * (1 + 速度係數 * 速度影響)
    jump_distance = BASE_JUMP_DISTANCE * (1 + (speed - 6) * 0.1)
    
    # 限制最小和最大跳躍距離，避免過早或過晚跳躍
    jump_distance = max(100, min(jump_distance, 200))
    
    return jump_distance


def jump():
    """
    執行跳躍動作
    
    說明：按下空白鍵或上箭頭鍵讓恐龍跳躍
    """
    pyg.press('space')
    # 或者使用 pyg.keyDown('up'); pyg.keyUp('up')


def duck():
    """
    執行蹲下動作
    
    說明：按下下箭頭鍵讓恐龍蹲下，躲避高空障礙物（翼手龍）
    """
    pyg.keyDown('down')
    time.sleep(0.4)  # 蹲下持續一段時間，確保能躲過翼手龍
    pyg.keyUp('down')


def start_game():
    """
    開始遊戲
    
    說明：在 Chrome Dino 頁面按下空白鍵開始遊戲
    """
    print("準備開始遊戲... 請將焦點切換到 Chrome 小恐龍遊戲頁面")
    time.sleep(3)  # 給使用者 3 秒時間切換到遊戲視窗
    pyg.press('space')  # 按下空白鍵開始遊戲
    time.sleep(0.5)
    print("遊戲已開始！自動遊玩中...")


# 全域變數：記錄遊戲開始時間，用於估算速度
game_start_time = time.time()

## 主程式邏輯

主迴圈會持續偵測障礙物，並根據障礙物類型和距離自動決定動作。

In [73]:
# ==================== 主程式 ====================

def auto_play_dino(max_iterations=None, debug=False):
    """
    自動遊玩 Chrome 小恐龍遊戲的主函式
    
    邏輯說明：
    1. 持續截取遊戲畫面
    2. 偵測前方的障礙物（仙人掌、翼手龍）
    3. 獲取當前遊戲速度，動態計算跳躍時機
    4. 根據障礙物類型做出決策：
       - 地面障礙物（仙人掌）或低空翼手龍 -> 跳躍
       - 高空翼手龍 -> 蹲下
    5. 避免重複動作（設定冷卻時間）
    
    參數：
        max_iterations: 最大執行次數，None 表示無限執行（直到手動中斷）
        debug: 是否顯示偵測資訊（用於除錯調整參數）
    
    注意：
        將滑鼠移動到螢幕左上角可緊急停止程式（pyautogui.FAILSAFE）
    """
    global game_start_time
    game_start_time = time.time()  # 重置遊戲開始時間
    
    start_game()  # 開始遊戲
    
    iterations = 0
    last_action_time = 0  # 上次動作的時間，用於避免重複動作
    action_cooldown = 0.3  # 動作冷卻時間（秒）
    
    print("\n開始自動偵測與反應...")
    print("按 Ctrl+C 或將滑鼠移至左上角可停止程式\n")
    
    try:
        while True:
            # 檢查是否達到最大執行次數
            if max_iterations is not None and iterations >= max_iterations:
                print(f"\n已達到最大執行次數 ({max_iterations})，停止自動遊玩")
                break
            
            iterations += 1
            current_time = time.time()
            
            # 1. 獲取當前遊戲速度並計算跳躍距離
            game_speed = get_game_speed()
            jump_trigger_distance = calculate_jump_distance(game_speed)
            
            # 2. 截取畫面並偵測障礙物
            screenshot = get_game_screenshot()
            obstacles = detect_obstacles(screenshot)
            
            # 3. 處理最近的障礙物
            if obstacles and len(obstacles) > 0:
                # 只處理最近的障礙物（距離最近的優先）
                nearest_obstacle = obstacles[0]
                obs_distance = nearest_obstacle['x']
                obs_type = nearest_obstacle['type']
                
                # 除錯模式：顯示偵測資訊
                if debug:
                    print(f"[偵測] 距離: {obs_distance:.1f}px, 類型: {obs_type}, 速度: {game_speed:.2f}")
                
                # 4. 根據障礙物類型和距離決定動作（需考慮動作冷卻時間）
                if current_time - last_action_time > action_cooldown:
                    
                    # 情況 1：高空翼手龍 -> 需要蹲下躲避
                    if obs_type == 'pterodactyl_high':
                        if obs_distance < 150:  # 在適當距離內開始蹲下
                            if debug:
                                print("  -> 偵測到高空翼手龍，執行蹲下動作！")
                            duck()
                            last_action_time = current_time
                            continue  # 動作執行後跳過此次迴圈，避免立即做出其他動作
                    
                    # 情況 2：地面障礙物（仙人掌）或低空翼手龍 -> 需要跳躍躲避
                    elif obs_type == 'cactus' or obs_type == 'pterodactyl_low':
                        # 根據遊戲速度動態調整跳躍時機
                        if obs_distance < jump_trigger_distance:
                            if debug:
                                print(f"  -> 偵測到{obs_type}，距離 {obs_distance:.1f} < {jump_trigger_distance:.1f}，執行跳躍動作！")
                            jump()
                            last_action_time = current_time
                            continue  # 動作執行後跳過此次迴圈
                
                # 若障礙物已經非常接近但還沒動作（安全保險）
                elif obs_distance < 80 and (obs_type == 'cactus' or obs_type == 'pterodactyl_low'):
                    if debug:
                        print("  -> 緊急跳躍！（障礙物過近）")
                    jump()
                    last_action_time = current_time
                    continue
            
            # 短暫延遲，避免過度消耗 CPU
            time.sleep(0.01)
            
    except KeyboardInterrupt:
        print("\n\n使用者中斷程式，停止自動遊玩")
    except Exception as e:
        print(f"\n\n發生錯誤：{e}")
        print("請檢查遊戲視窗是否正確開啟、參數設定是否合適")


def test_detection(duration=5):
    """
    測試障礙物偵測功能（不會自動操作，只顯示偵測結果）
    
    用途：
    在調整參數前，先用這個函式測試偵測是否正常運作
    確認能正確偵測到仙人掌和翼手龍，且距離計算正確
    
    參數：
        duration: 測試持續時間（秒）
    """
    print(f"開始測試偵測功能，將持續 {duration} 秒...")
    print("請開啟 Chrome 小恐龍遊戲並讓它執行")
    time.sleep(2)
    
    start_time = time.time()
    try:
        while time.time() - start_time < duration:
            screenshot = get_game_screenshot()
            obstacles = detect_obstacles(screenshot)
            
            if obstacles:
                print(f"[測試偵測] 找到 {len(obstacles)} 個障礙物:")
                for i, obs in enumerate(obstacles):
                    print(f"  {i+1}. 類型={obs['type']}, 距離={obs['x']:.1f}px, 位置(y)={obs['y']}")
            else:
                print("[測試偵測] 未偵測到障礙物")
            
            time.sleep(0.2)
        print("\n測試結束！")
    except KeyboardInterrupt:
        print("\n測試中斷")

## 使用說明

### 步驟 1：調整參數
根據你的螢幕解析度調整上方的參數設定區塊。最關鍵的是 `DETECT_REGION_LEFT`, `DETECT_REGION_TOP` 等座標。

### 步驟 2：測試偵測
先用測試功能確認能正確偵測障礙物，再開始自動遊玩。

In [74]:
# 測試障礙物偵測（建議先執行這個來調整參數）
# 開啟 Chrome 小恐龍遊戲（chrome://dino 或離線頁面）後再執行
test_detection(duration=10)

開始測試偵測功能，將持續 10 秒...
請開啟 Chrome 小恐龍遊戲並讓它執行
[測試偵測] 找到 3 個障礙物:
  1. 類型=pterodactyl_low, 距離=279.0px, 位置(y)=0
  2. 類型=pterodactyl_low, 距離=484.0px, 位置(y)=0
  3. 類型=pterodactyl_low, 距離=653.0px, 位置(y)=67
[測試偵測] 找到 3 個障礙物:
  1. 類型=pterodactyl_low, 距離=279.0px, 位置(y)=0
  2. 類型=pterodactyl_low, 距離=484.0px, 位置(y)=0
  3. 類型=pterodactyl_low, 距離=653.0px, 位置(y)=67
[測試偵測] 找到 3 個障礙物:
  1. 類型=pterodactyl_low, 距離=279.0px, 位置(y)=0
  2. 類型=pterodactyl_low, 距離=484.0px, 位置(y)=0
  3. 類型=pterodactyl_low, 距離=653.0px, 位置(y)=67
[測試偵測] 找到 3 個障礙物:
  1. 類型=pterodactyl_low, 距離=279.0px, 位置(y)=0
  2. 類型=pterodactyl_low, 距離=484.0px, 位置(y)=0
  3. 類型=pterodactyl_low, 距離=653.0px, 位置(y)=67
[測試偵測] 找到 3 個障礙物:
  1. 類型=pterodactyl_low, 距離=279.0px, 位置(y)=0
  2. 類型=pterodactyl_low, 距離=484.0px, 位置(y)=0
  3. 類型=pterodactyl_low, 距離=653.0px, 位置(y)=67
[測試偵測] 找到 3 個障礙物:
  1. 類型=pterodactyl_low, 距離=279.0px, 位置(y)=0
  2. 類型=pterodactyl_low, 距離=484.0px, 位置(y)=0
  3. 類型=pterodactyl_low, 距離=653.0px, 位置(y)=67
[測試偵測] 找到 3 個障礙物:
  1. 類型=pteroda

### 步驟 3：開始自動遊玩

參數說明：
- `max_iterations=None`：無限執行，直到手動中斷
- `debug=True`：顯示詳細偵測與決策資訊（調試用）
- `debug=False`：正常模式，只執行動作（較安靜）

In [75]:
# 開始自動遊玩
# 注意：執行前請將焦點切換到 Chrome 小恐龍遊戲視窗
# 將滑鼠移動到螢幕左上角可緊急中斷程式

auto_play_dino(max_iterations=None, debug=False)

準備開始遊戲... 請將焦點切換到 Chrome 小恐龍遊戲頁面
遊戲已開始！自動遊玩中...

開始自動偵測與反應...
按 Ctrl+C 或將滑鼠移至左上角可停止程式



使用者中斷程式，停止自動遊玩


## 除錯與調整建議

### 如果跳躍太早或太晚
- **跳躍太晚（撞到障礙物）**：增加 `BASE_JUMP_DISTANCE` 或 `SPEED_FACTOR`
- **跳躍太早（跳得太高影響後續動作）**：減少 `BASE_JUMP_DISTANCE` 或 `SPEED_FACTOR`

### 如果無法偵測到障礙物
- **完全沒偵測到**：調整 `OBSTACLE_THRESHOLD`（數值越低越容易偵測到暗色物件）
- **偵測到太多雜訊**：提高 `OBSTACLE_THRESHOLD` 或調整 `DETECT_REGION` 大小
- **偵測區域不對**：根據你的螢幕調整 `DETECT_REGION_LEFT`, `DETECT_REGION_TOP`

### 如果蹲下時機不對
- **高空翼手龍沒蹲下**：調整 `HIGH_PTERODACTYL_Y` 的值
- **低空翼手龍誤判為要蹲下**：提高 `HIGH_PTERODACTYL_Y`

### 速度調整說明
程式會根據遊玩時間自動估算速度：`Runner.instance_.currentSpeed` 會隨時間增加。
跳躍距離會隨速度動態調整：速度越快，`jump_trigger_distance` 越大，越早開始跳躍。